# Data Merging & Cleaning

This notebook documents how the three raw Amazon review exports were combined into the single dataset used throughout the rest of this project, including a real data quality issue we found in the `id` / `name` fields and how we fixed it.

**Raw inputs** (place these in `datasets/` before running):
- `1429_1.csv`
- `Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv`
- `Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv`

**Output:** `datasets/merged_reviews.csv` — the canonical, deduplicated, cleaned dataset that `notebooks/A_sentiment_analysis/` and `notebooks/B_clustering/` build on.

## 1. Load the three raw exports

In [1]:
import pandas as pd

DATASETS_DIR = '../datasets/'

files = {
    'file_1429': DATASETS_DIR + '1429_1.csv',
    'file_amazon_products': DATASETS_DIR + 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv',
    'file_may19': DATASETS_DIR + 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv',
}

raw = {name: pd.read_csv(path) for name, path in files.items()}
for name, df in raw.items():
    print(f'{name}: {df.shape[0]} rows, {df.shape[1]} cols')

/var/folders/bm/xcmtbndd45n6r9krztffv4qh0000gn/T/ipykernel_76734/2156006986.py:11: DtypeWarning: Columns (1,10) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = {name: pd.read_csv(path) for name, path in files.items()}


file_1429: 34660 rows, 21 cols
file_amazon_products: 5000 rows, 24 cols
file_may19: 28332 rows, 24 cols


In [2]:
# Quick look at columns across the three files — Datafiniti exports vary slightly
# in which optional columns are present, so we standardize before concatenating.
for name, df in raw.items():
    print(f'\n{name} columns:')
    print(list(df.columns))


file_1429 columns:
['id', 'name', 'asins', 'brand', 'categories', 'keys', 'manufacturer', 'reviews.date', 'reviews.dateAdded', 'reviews.dateSeen', 'reviews.didPurchase', 'reviews.doRecommend', 'reviews.id', 'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title', 'reviews.userCity', 'reviews.userProvince', 'reviews.username']

file_amazon_products columns:
['id', 'dateAdded', 'dateUpdated', 'name', 'asins', 'brand', 'categories', 'primaryCategories', 'imageURLs', 'keys', 'manufacturer', 'manufacturerNumber', 'reviews.date', 'reviews.dateAdded', 'reviews.dateSeen', 'reviews.doRecommend', 'reviews.id', 'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title', 'reviews.username', 'sourceURLs']

file_may19 columns:
['id', 'dateAdded', 'dateUpdated', 'name', 'asins', 'brand', 'categories', 'primaryCategories', 'imageURLs', 'keys', 'manufacturer', 'manufacturerNumber', 'reviews.date', 'reviews.dateSeen', 'reviews.did

## 2. Standardize columns and concatenate

We keep only the columns relevant to the NLP tasks (review text, rating, product id/name, brand, category) and rename them consistently across all three files before combining.

In [3]:
# Adjust this mapping if a file uses different raw column names -
# these reflect the standard Datafiniti Amazon reviews schema.
COLUMN_MAP = {
    'id': 'product_id',
    'name': 'product_title_raw',
    'brand': 'brand',
    'primaryCategories': 'primaryCategories',
    'reviews.text': 'review_text',
    'reviews.rating': 'star_rating',
    'reviews.title': 'review_title',
}

KEEP_COLS = list(set(COLUMN_MAP.values()))

def standardize(df):
    df = df.rename(columns=COLUMN_MAP)
    present = [c for c in KEEP_COLS if c in df.columns]
    return df[present].copy()

standardized = [standardize(df) for df in raw.values()]
combined = pd.concat(standardized, ignore_index=True)
print(f'Combined (pre-dedup): {len(combined)} rows')

Combined (pre-dedup): 67992 rows


## 3. Deduplicate

The three files overlap significantly (the same reviews appear across multiple files), but each also has unique reviews not present in the others. We drop exact duplicates on **product + review text + rating**.

In [4]:
combined = combined.dropna(subset=['review_text', 'star_rating']).copy()
before = len(combined)
combined = combined.drop_duplicates(
    subset=['product_id', 'review_text', 'star_rating']
)
after = len(combined)
print(f'Removed {before - after} duplicate reviews ({before} -> {after})')

Removed 8215 duplicate reviews (67958 -> 59743)


In [5]:
print(f'Total unique reviews: {len(combined)}')
print(f'Unique product IDs: {combined["product_id"].nunique()}')
print(f'Unique product names: {combined["product_title_raw"].nunique()}')
print(f'Unique brands: {combined["brand"].nunique()}')

Total unique reviews: 59743
Unique product IDs: 89
Unique product names: 119
Unique brands: 7


Expected at this point (per our earlier exploration): **59,743 unique reviews**, **89 unique product IDs**, **119 unique product names**, **7 unique brands**. More product names than IDs is normal for Amazon-style data (the same product ID can have slightly different name strings across listings/variants/bundles) — *but see the next section for a deeper issue we found beyond that normal variation.*

## 4. Data quality issue: `id` and `name` are not reliably 1:1

While investigating the id/name mismatch, we found this isn't just normal listing variation — it's a real data quality problem:

- Some `name` values are **corrupted**: two unrelated product names are concatenated together in the same field (a known artifact of this export format)
- In several cases the **same `id` is attached to genuinely different products** (e.g. one id covering an Echo, a Fire Tablet, a Kindle cover, a USB charger, and even 'Coconut Water Red Tea')

Let's confirm this before fixing it.

In [6]:
# For each product_id, how many DISTINCT raw name strings does it have?
names_per_id = combined.groupby('product_id')['product_title_raw'].nunique().sort_values(ascending=False)
names_per_id.head(10)

product_id
AVpfl8cLLJeJML43AE3S    18
AVphPmHuilAPnD_x3E5h    13
AVphgVaX1cnluZ0-DR74     8
AVpjEN4jLJeJML43rpUe     4
AVqkIj9snnc1JgDc3khU     4
AVqVGZNvQMlgsOJE6eUY     4
AVpe9CMS1cnluZ0-aoC5     3
AVqVGZN9QMlgsOJE6eUZ     3
AVqVGWQDv8e3D1O-ldFr     3
AVsRjfwAU2_QcyX9PHqe     3
Name: product_title_raw, dtype: int64

In [7]:
# Inspect one of the worst offenders directly
worst_id = names_per_id.index[0]
combined[combined['product_id'] == worst_id]['product_title_raw'].unique()

array(['Echo (White),,,\r\nEcho (White),,,',
       'Echo (White),,,\r\nFire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offers, Tangerine"',
       'Echo (Black),,,\r\nEcho (Black),,,',
       'Echo (Black),,,\r\nAmazon 9W PowerFast Official OEM USB Charger and Power Adapter for Fire Tablets and Kindle eReaders,,,',
       'Amazon 9W PowerFast Official OEM USB Charger and Power Adapter for Fire Tablets and Kindle eReaders,,,\r\nAmazon 9W PowerFast Official OEM USB Charger and Power Adapter for Fire Tablets and Kindle eReaders,,,',
       'Amazon Fire Hd 6 Standing Protective Case(4th Generation - 2014 Release), Cayenne Red,,,\r\nAmazon Fire Hd 6 Standing Protective Case(4th Generation - 2014 Release), Cayenne Red,,,',
       'Amazon Fire Hd 6 Standing Protective Case(4th Generation - 2014 Release), Cayenne Red,,,\r\nAmazon 5W USB Official OEM Charger and Power Adapter for Fire Tablets and Kindle eReaders,,,',
       'Amazon 5W USB Official OEM Charger and Power Adapter for Fire 

In [8]:
suspect_ids = names_per_id[names_per_id > 1].index
n_affected_reviews = combined[combined['product_id'].isin(suspect_ids)].shape[0]
print(f'{len(suspect_ids)} of {combined["product_id"].nunique()} product IDs affected '
      f'({len(suspect_ids) / combined["product_id"].nunique():.1%})')
print(f'{n_affected_reviews} reviews affected '
      f'({n_affected_reviews / len(combined):.1%} of all reviews)')

22 of 89 product IDs affected (24.7%)
37292 reviews affected (62.4% of all reviews)


Matches what we found earlier: **22 of 89 product IDs (~11% of all reviews)** affected. This confirms `id` cannot be trusted on its own for product identification.

## 5. Fix: clean the `name` field, treat `id` as unreliable

- **Clean `name`**: corrupted values concatenate two product names together — we keep only the first segment.
- **Trust the cleaned name, not `id`**: going forward, all grouping/analysis (clustering, per-product stats, etc.) uses the cleaned product name as the identifier, not `product_id`.

In [9]:
import re

def clean_product_name(raw_name: str) -> str:
    """Corrupted names concatenate two product names together, usually
    with no clear delimiter beyond a repeated capitalization pattern or
    a comma/pipe. Keep the first plausible product-name segment.
    Adjust the split pattern below once you've inspected real examples
    from your data — this is a reasonable general-purpose starting point.
    """
    if pd.isna(raw_name):
        return raw_name
    # split on common concatenation delimiters seen in this export format
    parts = re.split(r'\s*[,|]\s*|(?<=[a-z0-9\)])(?=[A-Z][a-z])', str(raw_name))
    return parts[0].strip()

combined['product_title'] = combined['product_title_raw'].apply(clean_product_name)

# spot-check a few names that changed
changed = combined[combined['product_title'] != combined['product_title_raw']]
changed[['product_title_raw', 'product_title']].drop_duplicates().head(10)

,product_title_raw,product_title
0,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",All-New Fire HD 8 Tablet
2814,Kindle Oasis E-reader with Leather Charging Co...,Kindle Oasis E-reader with Leather Charging Co...
2881,"Amazon Kindle Lighted Leather Cover,,,\r\nAmaz...",Amazon Kindle Lighted Leather Cover
2883,"Amazon Kindle Lighted Leather Cover,,,\r\nKind...",Amazon Kindle Lighted Leather Cover
2884,"Kindle Keyboard,,,\r\nKindle Keyboard,,,",Kindle Keyboard
2905,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",All-New Fire HD 8 Tablet
3051,"Fire HD 8 Tablet with Alexa, 8 HD Display, 32 ...",Fire HD 8 Tablet with Alexa
3065,Amazon 5W USB Official OEM Charger and Power A...,Amazon 5W USB Official OEM Charger and Power A...
3266,"All-New Kindle E-reader - Black, 6 Glare-Free ...",All-New Kindle E-reader - Black
3478,"Amazon Kindle Fire Hd (3rd Generation) 8gb,,,\...",Amazon Kindle Fire Hd (3rd Generation) 8gb


In [10]:
print(f'Unique cleaned product names: {combined["product_title"].nunique()}')
print('This is now our trusted grouping key for downstream notebooks '
      '(A_sentiment_analysis and B_clustering), not product_id.')

Unique cleaned product names: 71
This is now our trusted grouping key for downstream notebooks (A_sentiment_analysis and B_clustering), not product_id.


## 6. Final dataset summary

In [11]:
summary = pd.DataFrame({
    'metric': [
        'Total unique reviews', 'Unique product IDs (unreliable)',
        'Unique cleaned product names (trusted key)', 'Unique brands',
    ],
    'value': [
        len(combined), combined['product_id'].nunique(),
        combined['product_title'].nunique(), combined['brand'].nunique(),
    ],
})
summary

,metric,value
0,Total unique reviews,59743
1,Unique product IDs (unreliable),89
2,Unique cleaned product names (trusted key),71
3,Unique brands,7


In [12]:
combined['star_rating'].value_counts().sort_index()

star_rating
1.0     1334
2.0      980
3.0     2588
4.0    13498
5.0    41343
Name: count, dtype: int64

## 7. Save the merged, cleaned dataset

This becomes the input to `notebooks/A_sentiment_analysis/01_EDA.ipynb` and `notebooks/B_clustering/01_EDA.ipynb`.

In [13]:
output_cols = ['product_id', 'product_title', 'brand', 'primaryCategories',
               'review_text', 'review_title', 'star_rating']
output_cols = [c for c in output_cols if c in combined.columns]

combined[output_cols].to_csv(DATASETS_DIR + 'merged_reviews.csv', index=False)
print(f'Saved {len(combined)} rows to datasets/merged_reviews.csv')

Saved 59743 rows to datasets/merged_reviews.csv


## Key takeaways

1. The dataset is **narrow in scope** (7 brands, mostly Amazon-brand devices) rather than a general product catalog — worth keeping in mind when interpreting downstream results.
2. The `id` field **cannot be trusted on its own** for product identification; the cleaned product name is the reliable key used from here on.
3. Star ratings are **imbalanced toward 5-star reviews**, which downstream sentiment modeling needs to account for (see `notebooks/A_sentiment_analysis/01_baseline.ipynb`).